# MetaQuest VR SO100 Robot Control
## MetaQuest VR 헤드셋을 사용하여 SO100 로봇 제어

이 노트북은 MetaQuest VR 헤드셋을 사용하여 SO100 로봇을 제어하는 시스템입니다.

**3단계로 나눠서 실행:**
1. 🤖 **로봇 연결** - SO100 로봇과 통신 시작
2. 📱 **MetaQuest 접속** - VR 헤드셋 연결 준비  
3. 🎮 **제어 시작** - VR로 팔 조종 시작

In [1]:
import asyncio
import logging
import math
import os
import sys
import threading
import time
import traceback
import socket
from typing import Optional, Dict, Any

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Store global variables for use across cells
globals_dict = {
    'robot': None,
    'vr_monitor': None,
    'controller': None,
    'event_loop_thread': None,
    'event_loop': None,
}

# Helper class for managing asyncio event loop in a separate thread
class AsyncEventLoopThread(threading.Thread):
    """Thread that runs an asyncio event loop"""
    def __init__(self):
        super().__init__(daemon=True)
        self.loop = None
        self.ready = threading.Event()
        
    def run(self):
        """Run the event loop in this thread"""
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)
        self.ready.set()
        self.loop.run_forever()
    
    def stop(self):
        """Stop the event loop"""
        if self.loop and self.loop.is_running():
            self.loop.call_soon_threadsafe(self.loop.stop)
    
    def run_coroutine(self, coro):
        """Run a coroutine in the event loop"""
        if self.loop and self.loop.is_running():
            return asyncio.run_coroutine_threadsafe(coro, self.loop)
        return None

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# XLeVR Configuration
XLEVR_PATH = "/home/choyunsang/XLeRobot/XLeVR"

def get_local_ip():
    """Get local machine IP address"""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect(("8.8.8.8", 80))
            return s.getsockname()[0]
    except Exception:
        try:
            return socket.gethostbyname(socket.gethostname())
        except Exception:
            return "localhost"

def setup_xlevr_environment():
    """Setup XLeVR environment"""
    if XLEVR_PATH not in sys.path:
        sys.path.insert(0, XLEVR_PATH)
    os.chdir(XLEVR_PATH)
    os.environ['PYTHONPATH'] = f"{XLEVR_PATH}:{os.environ.get('PYTHONPATH', '')}"
    logger.info(f"✅ XLeVR environment configured: {XLEVR_PATH}")

def import_xlevr_modules():
    """Import XLeVR modules"""
    try:
        from xlevr.config import XLeVRConfig
        from xlevr.inputs.vr_ws_server import VRWebSocketServer
        from xlevr.inputs.base import ControlGoal, ControlMode
        logger.info("✅ XLeVR modules imported")
        return XLeVRConfig, VRWebSocketServer, ControlGoal, ControlMode
    except ImportError as e:
        logger.error(f"❌ Failed to import XLeVR modules: {e}")
        return None, None, None, None

# Import SSL and HTTP modules for HTTPS server
import ssl
import http.server

class SimpleAPIHandler(http.server.BaseHTTPRequestHandler):
    """HTTP request handler for serving web UI"""
    
    def end_headers(self):
        """Add CORS headers to all responses."""
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        try:
            super().end_headers()
        except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError, ssl.SSLError):
            pass
    
    def do_OPTIONS(self):
        """Handle preflight CORS requests."""
        self.send_response(200)
        self.end_headers()
    
    def log_message(self, format, *args):
        """Override to reduce HTTP request logging noise."""
        pass  # Disable default HTTP logging
    
    def do_GET(self):
        """Handle GET requests."""
        if self.path == '/' or self.path == '/index.html':
            self.serve_file('web-ui/index.html', 'text/html')
        elif self.path.endswith('.css'):
            self.serve_file(f'web-ui{self.path}', 'text/css')
        elif self.path.endswith('.js'):
            self.serve_file(f'web-ui{self.path}', 'application/javascript')
        elif self.path.endswith(('.jpg', '.jpeg', '.png', '.gif')):
            content_type = 'image/jpeg' if self.path.endswith(('.jpg', '.jpeg')) else 'image/png' if self.path.endswith('.png') else 'image/gif'
            self.serve_file(f'web-ui{self.path}', content_type)
        else:
            self.send_error(404, "Not found")
    
    def serve_file(self, filename, content_type):
        """Serve a file with the given content type."""
        try:
            file_path = os.path.join(XLEVR_PATH, filename)
            
            if os.path.exists(file_path):
                with open(file_path, 'rb') as f:
                    content = f.read()
                
                self.send_response(200)
                self.send_header('Content-Type', content_type)
                self.end_headers()
                self.wfile.write(content)
            else:
                self.send_error(404, f"File not found: {filename}")
        except Exception as e:
            logger.error(f"Error serving file {filename}: {e}")
            self.send_error(500, "Internal server error")

class SimpleHTTPSServer:
    """HTTPS server for providing web interface"""
    
    def __init__(self, host, port):
        self.host = host
        self.port = port
        self.httpd = None
        self.server_thread = None
    
    async def start(self):
        """Start the HTTPS server."""
        try:
            self.httpd = http.server.HTTPServer((self.host, self.port), SimpleAPIHandler)
            
            # Setup SSL
            context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
            context.load_cert_chain(
                os.path.join(XLEVR_PATH, 'cert.pem'),
                os.path.join(XLEVR_PATH, 'key.pem')
            )
            self.httpd.socket = context.wrap_socket(self.httpd.socket, server_side=True)
            
            # Start server in a separate thread
            self.server_thread = threading.Thread(target=self.httpd.serve_forever, daemon=True)
            self.server_thread.start()
            logger.info(f"✅ HTTPS server started on {self.host}:{self.port}")
            
        except Exception as e:
            logger.error(f"❌ Failed to start HTTPS server: {e}")
            raise
    
    async def stop(self):
        """Stop the HTTPS server."""
        if self.httpd:
            self.httpd.shutdown()
            if self.server_thread:
                self.server_thread.join(timeout=5)
            logger.info("✅ HTTPS server stopped")

# Setup environment on first import
setup_xlevr_environment()
print("✅ XLeVR environment configured")

2026-05-18 19:14:39,645 - __main__ - INFO - ✅ XLeVR environment configured: /home/choyunsang/XLeRobot/XLeVR


✅ XLeVR environment configured


In [3]:
# VR Monitor Class - for SO100
class VRControlMonitor:
    """VR controller monitor for MetaQuest - SO100 single arm"""
    
    def __init__(self):
        self.config = None
        self.vr_server = None
        self.https_server = None
        self.is_running = False
        self.right_goal = None
        self._goal_lock = threading.Lock()
        self.command_queue = None
        self.monitoring_task = None
        self.last_update_time = time.time()
        
    def initialize(self):
        """Initialize VR monitor - SO100 single arm"""
        logger.info("🔧 Initializing VR Monitor (SO100)...")
        
        XLeVRConfig, VRWebSocketServer, _, _ = import_xlevr_modules()
        if XLeVRConfig is None:
            logger.error("❌ Failed to import XLeVR modules")
            return False
        
        self.config = XLeVRConfig()
        self.config.enable_vr = True
        self.config.enable_keyboard = False
        self.command_queue = asyncio.Queue()
        
        try:
            # Create VR WebSocket server
            self.vr_server = VRWebSocketServer(
                command_queue=self.command_queue,
                config=self.config,
                print_only=False
            )
            logger.info("✅ VR WebSocket server created")
            
            # Create HTTPS web UI server
            host_ip = self.config.host_ip if self.config.host_ip != "0.0.0.0" else "0.0.0.0"
            https_port = self.config.https_port if hasattr(self.config, 'https_port') else 8443
            
            self.https_server = SimpleHTTPSServer(host_ip, https_port)
            logger.info("✅ HTTPS web server created")
            
            # Display connection info
            host_ip_display = get_local_ip() if self.config.host_ip == "0.0.0.0" else self.config.host_ip
            websocket_port = self.config.websocket_port if hasattr(self.config, 'websocket_port') else 8442
            
            print("\n" + "="*70)
            print("📱 MetaQuest VR Headset - COPY THIS URL:")
            print(f"\n   https://{host_ip_display}:{https_port}\n")
            print(f"Connection Details:")
            print(f"  • HTTPS Web UI: {host_ip_display}:{https_port}")
            print(f"  • WebSocket: {host_ip_display}:{websocket_port}")
            print("="*70)
            print("🎮 Mode: SO100 Single Arm Control")
            print("="*70 + "\n")
            
            return True
        except Exception as e:
            logger.error(f"❌ Failed to initialize servers: {e}")
            traceback.print_exc()
            return False
    
    async def monitor_commands_async(self):
        """Monitor VR commands asynchronously - right arm only"""
        logger.info("🔄 VR command monitoring started")
        while self.is_running:
            try:
                # Non-blocking get with timeout
                goal = await asyncio.wait_for(self.command_queue.get(), timeout=0.1)
                # Only process RIGHT ARM commands
                if hasattr(goal, 'arm') and goal.arm == "right":
                    with self._goal_lock:
                        self.right_goal = goal
                        self.last_update_time = time.time()
                        logger.debug(f"📍 VR update received")
            except asyncio.TimeoutError:
                continue
            except Exception as e:
                logger.error(f"❌ Command processing error: {e}")
                await asyncio.sleep(0.01)
    
    async def start_monitoring_async(self):
        """Start monitoring VR input (async version)"""
        try:
            # Start HTTPS web server
            logger.info("🌐 Starting HTTPS web server...")
            await self.https_server.start()
            
            # Start VR WebSocket server
            logger.info("🔌 Starting VR WebSocket server...")
            await self.vr_server.start()
            
            self.is_running = True
            logger.info("✅ VR monitoring started")
            await self.monitor_commands_async()
        except Exception as e:
            logger.error(f"❌ VR monitoring error: {e}")
            traceback.print_exc()
        finally:
            await self.stop_monitoring_async()
    
    async def stop_monitoring_async(self):
        """Stop monitoring (async version)"""
        self.is_running = False
        if self.https_server:
            await self.https_server.stop()
        if self.vr_server:
            await self.vr_server.stop()
        logger.info("✅ VR Monitor stopped")
    
    def get_right_goal_nowait(self):
        """Get current right arm goal"""
        with self._goal_lock:
            return self.right_goal
    
    def is_connected(self):
        """Check if VR client is connected"""
        if self.vr_server:
            return len(self.vr_server.clients) > 0
        return False

print("✅ VR Monitor class defined")

✅ VR Monitor class defined


In [4]:
# SO100 VR Controller Class
class SO100VRController:
    """SO100 arm VR teleoperation control"""
    
    def __init__(self, kp=0.5):
        self.kp = kp
        self.current_x = 0.1629
        self.current_y = 0.1131
        self.pitch = 0.0
        
        self.prev_vr_pos = None
        self.prev_wrist_flex = None
        self.prev_wrist_roll = None
        
        # Joint calibration coefficients
        self.joint_calibration = {
            'shoulder_pan': [6.0, 1.0],
            'shoulder_lift': [2.0, 0.97],
            'elbow_flex': [0.0, 1.05],
            'wrist_flex': [0.0, 0.94],
            'wrist_roll': [0.0, 0.5],
            'gripper': [0.0, 1.0],
        }
        
        self.target_positions = {
            "shoulder_pan": 0.0,
            "shoulder_lift": 0.0,
            "elbow_flex": 0.0,
            "wrist_flex": 0.0,
            "wrist_roll": 0.0,
            "gripper": 0.0,
        }

    def apply_joint_calibration(self, joint_name, raw_position):
        """Apply joint calibration coefficients"""
        if joint_name in self.joint_calibration:
            offset, scale = self.joint_calibration[joint_name]
            return (raw_position - offset) * scale
        return raw_position

    def inverse_kinematics(self, x, y, l1=0.1159, l2=0.1350):
        """Calculate inverse kinematics for 2-link arm"""
        theta1_offset = math.atan2(0.028, 0.11257)
        theta2_offset = math.atan2(0.0052, 0.1349) + theta1_offset
        
        r = math.sqrt(x**2 + y**2)
        r_max = l1 + l2
        
        if r > r_max:
            scale_factor = r_max / r
            x *= scale_factor
            y *= scale_factor
            r = r_max
        
        r_min = abs(l1 - l2)
        if r < r_min and r > 0:
            scale_factor = r_min / r
            x *= scale_factor
            y *= scale_factor
            r = r_min
        
        cos_theta2 = -(r**2 - l1**2 - l2**2) / (2 * l1 * l2)
        cos_theta2 = max(-1.0, min(1.0, cos_theta2))
        
        theta2 = math.pi - math.acos(cos_theta2)
        
        beta = math.atan2(y, x)
        gamma = math.atan2(l2 * math.sin(theta2), l1 + l2 * math.cos(theta2))
        theta1 = beta + gamma
        
        joint2 = theta1 + theta1_offset
        joint3 = theta2 + theta2_offset
        
        joint2 = max(-0.1, min(3.45, joint2))
        joint3 = max(-0.2, min(math.pi, joint3))
        
        joint2_deg = math.degrees(joint2)
        joint3_deg = math.degrees(joint3)
        
        joint2_deg = 90 - joint2_deg
        joint3_deg = joint3_deg - 90
        
        return joint2_deg, joint3_deg

    def move_to_zero_position(self, robot, duration=3.0):
        """Move arm to zero position using P control"""
        logger.info("[SO100] Moving to Zero Position...")
        
        zero_positions = {
            'shoulder_pan': 0.0,
            'shoulder_lift': 0.0,
            'elbow_flex': 0.0,
            'wrist_flex': 0.0,
            'wrist_roll': 0.0,
            'gripper': 0.0
        }
        
        control_freq = 50
        total_steps = int(duration * control_freq)
        step_time = 1.0 / control_freq
        
        for step in range(total_steps):
            current_obs = robot.get_observation()
            current_positions = {}
            
            for key, value in current_obs.items():
                if key.endswith('.pos'):
                    motor_name = key.removesuffix('.pos')
                    calibrated_value = self.apply_joint_calibration(motor_name, value)
                    current_positions[motor_name] = calibrated_value
            
            robot_action = {}
            for joint_name, target_pos in zero_positions.items():
                if joint_name in current_positions:
                    current_pos = current_positions[joint_name]
                    error = target_pos - current_pos
                    control_output = self.kp * error
                    new_position = current_pos + control_output
                    robot_action[f"{joint_name}.pos"] = new_position
            
            if robot_action:
                robot.send_action(robot_action)
            
            if step % (control_freq // 2) == 0:
                progress = (step / total_steps) * 100
                logger.info(f"Moving to zero position: {progress:.1f}%")
            
            time.sleep(step_time)
        
        logger.info("✅ Robot moved to zero position")

    def handle_vr_input(self, vr_goal):
        """Handle VR input with delta action control - incremental position updates"""
        if vr_goal is None or not hasattr(vr_goal, 'target_position') or vr_goal.target_position is None:
            return
        
        current_vr_pos = vr_goal.target_position
        
        # Initialize previous VR position if not set
        if self.prev_vr_pos is None:
            self.prev_vr_pos = current_vr_pos
            return  # Skip first frame to establish baseline
        
        # Calculate relative change (delta) from previous frame
        vr_x = (current_vr_pos[0] - self.prev_vr_pos[0]) * 220  # Scale for shoulder pan
        vr_y = (current_vr_pos[1] - self.prev_vr_pos[1]) * 70
        vr_z = (current_vr_pos[2] - self.prev_vr_pos[2]) * 70
        
        # Update previous position for next frame
        self.prev_vr_pos = current_vr_pos
        
        # Delta control parameters - adjust these for sensitivity
        pos_scale = 0.01  # Position sensitivity scaling
        angle_scale = 4.0  # Angle sensitivity scaling
        delta_limit = 0.01  # Maximum delta per update (meters)
        angle_limit = 8.0  # Maximum angle delta per update (degrees)
        
        delta_x = vr_x * pos_scale
        delta_y = vr_y * pos_scale
        delta_z = vr_z * pos_scale
        
        # Limit delta values to prevent sudden movements
        delta_x = max(-delta_limit, min(delta_limit, delta_x))
        delta_y = max(-delta_limit, min(delta_limit, delta_y))
        delta_z = max(-delta_limit, min(delta_limit, delta_z))
        
        # Update end-effector position incrementally (relative position)
        self.current_x += -delta_z  # VR Z maps to robot x
        self.current_y += delta_y   # VR Y maps to robot y
        
        # Handle wrist angles with delta control - use relative changes
        if hasattr(vr_goal, 'wrist_flex_deg') and vr_goal.wrist_flex_deg is not None:
            # Initialize previous wrist_flex if not set
            if self.prev_wrist_flex is None:
                self.prev_wrist_flex = vr_goal.wrist_flex_deg
            else:
                # Calculate relative change from previous frame
                delta_pitch = (vr_goal.wrist_flex_deg - self.prev_wrist_flex) * angle_scale
                delta_pitch = max(-angle_limit, min(angle_limit, delta_pitch))
                self.pitch += delta_pitch
                self.pitch = max(-90, min(90, self.pitch))  # Limit pitch range
                
                # Update previous value for next frame
                self.prev_wrist_flex = vr_goal.wrist_flex_deg
        
        if hasattr(vr_goal, 'wrist_roll_deg') and vr_goal.wrist_roll_deg is not None:
            # Initialize previous wrist_roll if not set
            if self.prev_wrist_roll is None:
                self.prev_wrist_roll = vr_goal.wrist_roll_deg
            else:
                delta_roll = (vr_goal.wrist_roll_deg - self.prev_wrist_roll) * angle_scale
                delta_roll = max(-angle_limit, min(angle_limit, delta_roll))
                
                current_roll = self.target_positions.get("wrist_roll", 0.0)
                new_roll = current_roll + delta_roll
                new_roll = max(-90, min(90, new_roll))  # Limit roll range
                self.target_positions["wrist_roll"] = new_roll
                
                # Update previous value for next frame
                self.prev_wrist_roll = vr_goal.wrist_roll_deg
        
        # VR X axis controls shoulder_pan joint (delta control)
        if abs(delta_x) > 0.001:  # Only update if significant movement
            x_scale = 200.0  # Scaling factor for delta control
            delta_pan = delta_x * x_scale
            delta_pan = max(-angle_limit, min(angle_limit, delta_pan))
            current_pan = self.target_positions.get("shoulder_pan", 0.0)
            new_pan = current_pan + delta_pan
            new_pan = max(-180, min(180, new_pan))  # Limit pan range
            self.target_positions["shoulder_pan"] = new_pan
        
        # Inverse kinematics for relative position control
        try:
            joint2_target, joint3_target = self.inverse_kinematics(self.current_x, self.current_y)
            # Smooth transition to new joint positions
            alpha = 0.1  # Smoothing factor (lower = smoother)
            self.target_positions["shoulder_lift"] = (
                (1 - alpha) * self.target_positions.get("shoulder_lift", 0.0) + 
                alpha * joint2_target
            )
            self.target_positions["elbow_flex"] = (
                (1 - alpha) * self.target_positions.get("elbow_flex", 0.0) + 
                alpha * joint3_target
            )
        except Exception as e:
            logger.debug(f"IK calculation error: {e}")
        
        # Calculate wrist_flex to maintain end-effector orientation
        self.target_positions["wrist_flex"] = (
            -self.target_positions["shoulder_lift"] - 
            self.target_positions["elbow_flex"] + 
            self.pitch
        )
        
        # Gripper control
        if hasattr(vr_goal, 'metadata') and vr_goal.metadata.get('trigger', 0) > 0.5:
            self.target_positions["gripper"] = 45
        else:
            self.target_positions["gripper"] = 0.0

    def p_control_action(self, robot):
        """Generate P-control action"""
        obs = robot.get_observation()
        action = {}
        
        for joint_name, target_pos in self.target_positions.items():
            current_key = f"{joint_name}.pos"
            if current_key in obs:
                current = obs[current_key]
                # Don't apply calibration when calculating error
                error = target_pos - current
                control = self.kp * error
                action[f"{joint_name}.pos"] = current + control
        
        return action

print("✅ SO100 VR Controller class defined")

✅ SO100 VR Controller class defined


---

## 🤖 SECTION 1: 로봇 연결

**SO100 로봇을 연결하고 초기화합니다.**

✅ 확인사항:
- 로봇이 전원 ON되어 있나요?
- USB 포트가 연결되어 있나요? (기본값: /dev/ttyACM0)

In [5]:
# Connect to SO100 Robot
print("🔄 Connecting to SO100...")

try:
    from lerobot.robots.so_follower.so_follower import SO100Follower
    from lerobot.robots.so_follower.config_so_follower import SO100FollowerConfig
    
    # Use default port or specify manually
    port = "/dev/ttyACM1"
    
    robot_config = SO100FollowerConfig(port=port)
    robot = SO100Follower(robot_config)
    
    robot.connect()
    logger.info(f"✅ Robot connected successfully on {port}")
    
    if robot.is_calibrated:
        logger.info("✅ Robot is calibrated")
    else:
        logger.warning("⚠️  Robot needs calibration")
    
    # Store robot reference
    globals_dict['robot'] = robot
    
    # Initialize VR controller
    controller = SO100VRController(kp=0.5)
    
    # Move to zero position
    logger.info("🎯 Moving to zero position...")
    controller.move_to_zero_position(robot)
    
    globals_dict['controller'] = controller
    
    print("\n" + "="*70)
    print("✅ ROBOT CONNECTED AND READY")
    print("="*70)
    print("Next Step: Run SECTION 2 to setup MetaQuest connection")
    
except Exception as e:
    logger.error(f"❌ Robot connection failed: {e}")
    traceback.print_exc()

🔄 Connecting to SO100...


2026-05-18 19:14:40,938 - lerobot.robots.so_follower.so_follower - INFO - None SOFollower connected.
2026-05-18 19:14:40,939 - __main__ - INFO - ✅ Robot connected successfully on /dev/ttyACM1
2026-05-18 19:14:40,945 - __main__ - INFO - ✅ Robot is calibrated
2026-05-18 19:14:40,946 - __main__ - INFO - 🎯 Moving to zero position...
2026-05-18 19:14:40,946 - __main__ - INFO - [SO100] Moving to Zero Position...
2026-05-18 19:14:40,948 - __main__ - INFO - Moving to zero position: 0.0%
2026-05-18 19:14:41,490 - __main__ - INFO - Moving to zero position: 16.7%
2026-05-18 19:14:42,035 - __main__ - INFO - Moving to zero position: 33.3%
2026-05-18 19:14:42,578 - __main__ - INFO - Moving to zero position: 50.0%
2026-05-18 19:14:43,123 - __main__ - INFO - Moving to zero position: 66.7%
2026-05-18 19:14:43,670 - __main__ - INFO - Moving to zero position: 83.3%
2026-05-18 19:14:44,208 - __main__ - INFO - ✅ Robot moved to zero position



✅ ROBOT CONNECTED AND READY
Next Step: Run SECTION 2 to setup MetaQuest connection


---

## 📱 SECTION 2: MetaQuest 연결

**VR 서버를 시작하고 MetaQuest 헤드셋을 연결합니다.**

✅ 이 셀을 실행하면:
1. **HTTPS 웹 UI 서버** 시작 (포트 8443) - 웹 페이지 제공
2. **WebSocket 서버** 시작 (포트 8442) - VR 데이터 수신
3. **MetaQuest 접속 주소 표시** - 복사해서 헤드셋 브라우저에 입력
   - 🔗 URL 형식: `https://[IP]:[HTTPS_PORT]`
   - 예: `https://192.168.1.100:8443`

⚠️ **중요 - SSL 인증서 경고:**
- MetaQuest 브라우저에서 **"연결이 안전하지 않습니다"** 경고가 나올 수 있습니다
- 이는 **자체 서명된 인증서를 사용**하기 때문입니다 (정상입니다)
- **해결 방법:**
  - 🔐 "연결이 안전하지 않습니다" 화면 보임
  - ➡️ **"고급" 또는 "Advanced"** 클릭
  - ➡️ **"계속" 또는 "Continue"** 클릭
  - ➡️ 웹 페이지 로드 완료!

⏰ 주의: 이 셀은 **계속 실행 상태를 유지**합니다
- 헤드셋에서 접속 후, 다음 셀(SECTION 3)을 실행하세요
- 🟢 메시지에서 "MetaQuest Connected" 표시되면 준비 완료

In [ ]:
# Setup VR Monitor with proper asyncio event loop management
print("⏳ Starting VR Monitor (HTTPS + WebSocket)...")

try:
    # Stop previous event loop if it exists
    if globals_dict['event_loop_thread'] is not None:
        globals_dict['event_loop_thread'].stop()
        time.sleep(0.5)
    
    # Create and start a new event loop thread
    event_loop_thread = AsyncEventLoopThread()
    event_loop_thread.start()
    event_loop_thread.ready.wait(timeout=5)  # Wait for loop to be ready
    
    if event_loop_thread.loop is None:
        print("❌ Failed to create asyncio event loop")
    else:
        logger.info("✅ Asyncio event loop thread started")
        globals_dict['event_loop_thread'] = event_loop_thread
        globals_dict['event_loop'] = event_loop_thread.loop
        
        # Initialize VR monitor
        vr_monitor = VRControlMonitor()
        
        if not vr_monitor.initialize():
            print("❌ VR Monitor initialization failed")
        else:
            print("\n" + "="*70)
            print("✅ VR SERVERS READY")
            print("="*70)
            
            # Start VR monitoring in the event loop thread
            future = event_loop_thread.run_coroutine(vr_monitor.start_monitoring_async())
            
            globals_dict['vr_monitor'] = vr_monitor
            
            # Wait a moment for servers to start
            time.sleep(1)
            
            print("\n📋 CONNECTION GUIDE:")
            print("1. ⬆️  COPY the HTTPS URL from above")
            print("   (Should start with 'https://' NOT 'http://')")
            print("")
            print("2. 🥽 Open your MetaQuest headset browser")
            print("")
            print("3. 📍 Paste the URL into the address bar")
            print("")
            print("4. ⚠️  You may see 'Not secure' warning:")
            print("   → Look for 'Advanced' or 'More options' button")
            print("   → Click it to see more options")
            print("   → Click 'Proceed' or 'Continue anyway'")
            print("   → Wait for page to load")
            print("")
            print("5. ✅ Once page loads, run SECTION 3 to start control")
            print("\n⏳ Waiting for MetaQuest connection...")
            print("(This cell keeps running, that's normal)\n")
            
            # Wait for connection
            wait_time = 0
            connected = False
            while not connected and wait_time < 60:
                if vr_monitor.is_connected():
                    connected = True
                    break
                
                time.sleep(1)
                wait_time += 1
                
                if wait_time % 5 == 0:
                    print(f"🔄 Waiting for MetaQuest... ({wait_time}s)")
            
            if connected:
                print("\n" + "="*70)
                print("🎉 MetaQuest Connected!")
                print("="*70)
                print("✅ Ready for control - Run SECTION 3")
            else:
                print("\n" + "="*70)
                print("⏳ Still waiting for connection...")
                print("="*70)
                print("You can still proceed to SECTION 3")
                print("(Connection may happen later)")
        
except Exception as e:
    logger.error(f"❌ VR Monitor error: {e}")
    traceback.print_exc()

2026-05-18 19:14:44,225 - __main__ - INFO - ✅ Asyncio event loop thread started
2026-05-18 19:14:44,226 - __main__ - INFO - 🔧 Initializing VR Monitor (SO100)...
2026-05-18 19:14:44,409 - __main__ - INFO - ✅ XLeVR modules imported
2026-05-18 19:14:44,410 - __main__ - INFO - ✅ VR WebSocket server created
2026-05-18 19:14:44,410 - __main__ - INFO - ✅ HTTPS web server created
2026-05-18 19:14:44,411 - __main__ - INFO - 🌐 Starting HTTPS web server...


⏳ Starting VR Monitor (HTTPS + WebSocket)...

📱 MetaQuest VR Headset - COPY THIS URL:

   https://192.168.0.44:8443

Connection Details:
  • HTTPS Web UI: 192.168.0.44:8443
  • WebSocket: 192.168.0.44:8442
🎮 Mode: SO100 Single Arm Control


✅ VR SERVERS READY


2026-05-18 19:14:44,414 - __main__ - INFO - ✅ HTTPS server started on 0.0.0.0:8443
2026-05-18 19:14:44,414 - __main__ - INFO - 🔌 Starting VR WebSocket server...
2026-05-18 19:14:44,414 - xlevr.inputs.vr_ws_server - INFO - SSL certificate and key loaded successfully for WebSocket server
2026-05-18 19:14:44,421 - websockets.server - INFO - server listening on 0.0.0.0:8442
2026-05-18 19:14:44,422 - xlevr.inputs.vr_ws_server - INFO - VR WebSocket server running on wss://0.0.0.0:8442
2026-05-18 19:14:44,422 - __main__ - INFO - ✅ VR monitoring started
2026-05-18 19:14:44,422 - __main__ - INFO - 🔄 VR command monitoring started



📋 CONNECTION GUIDE:
1. ⬆️  COPY the HTTPS URL from above
   (Should start with 'https://' NOT 'http://')

2. 🥽 Open your MetaQuest headset browser

3. 📍 Paste the URL into the address bar

4. ⚠️  You may see 'Not secure' warning:
   → Look for 'Advanced' or 'More options' button
   → Click it to see more options
   → Click 'Proceed' or 'Continue anyway'
   → Wait for page to load

5. ✅ Once page loads, run SECTION 3 to start control

⏳ Waiting for MetaQuest connection...
(This cell keeps running, that's normal)

🔄 Waiting for MetaQuest... (5s)
🔄 Waiting for MetaQuest... (10s)
🔄 Waiting for MetaQuest... (15s)
🔄 Waiting for MetaQuest... (20s)


2026-05-18 19:15:05,815 - websockets.server - INFO - connection open
2026-05-18 19:15:05,816 - xlevr.inputs.vr_ws_server - INFO - VR client connected: ('192.168.0.65', 33814)
2026-05-18 19:15:05,855 - xlevr.inputs.vr_ws_server - INFO - 🎯 LEFT auto-activated - controlling left arm
2026-05-18 19:15:05,856 - xlevr.inputs.vr_ws_server - INFO - 🎯 RIGHT auto-activated - controlling right arm


...

[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.127], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: 

2026-05-18 19:15:37,095 - xlevr.inputs.vr_ws_server - INFO - 🤏 RIGHT gripper OPENED


[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: 

2026-05-18 19:15:37,262 - xlevr.inputs.vr_ws_server - INFO - 🤏 RIGHT gripper CLOSED


[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: [-0.561, 0.870, 0.128], Rotation: [5.5, 6.2, -176.2]
[VR_WS] Headset - Position: 

---

## 🎮 SECTION 3: VR 제어 시작

**MetaQuest 컨트롤러로 SO100 로봇을 제어합니다.**

✅ 사전 요구사항:
- ✔️ SECTION 1에서 로봇 연결 완료
- ✔️ SECTION 2에서 MetaQuest 접속 완료
- ✔️ MetaQuest 헤드셋이 연결되어 있음

🎮 컨트롤러 사용법:
- **위치 이동**: 컨트롤러 위치 이동 → 팔의 엔드이펙터 위치 변경
- **손목 회전**: 컨트롤러 손목 각도 → 손목 조정
- **그리퍼**: 트리거 버튼 → 그리퍼 개폐

🔄 **실시간 모니터링**:
- VR 입력과 로봇 제어가 동시에 실행됩니다
- 매 2초마다 현재 상태가 표시됩니다 (루프 주파수, VR 업데이트 빈도, 엔드이펙터 위치)
- 실제 VR 조종 데이터가 계속 받아집니다

⏹️ 종료: 셀 실행 중지 (Stop 버튼) → 로봇 안전 정지

In [7]:
# Main VR Control Loop
print("🚀 Starting VR Control Loop...")
print("💡 Press Stop button to exit\n")

robot = globals_dict.get('robot')
controller = globals_dict.get('controller')
vr_monitor = globals_dict.get('vr_monitor')

if not all([robot, controller, vr_monitor]):
    print("❌ Error: Missing robot, controller, or VR monitor")
    print("   Run SECTION 1 and SECTION 2 first!")
else:
    try:
        loop_count = 0
        start_time = time.time()
        vr_update_count = 0
        last_print_time = time.time()
        
        print("📊 Monitoring Status:")
        print("-" * 70)
        
        # Control loop
        while True:
            # Get VR input
            right_goal = vr_monitor.get_right_goal_nowait()
            
            if right_goal is not None:
                # Process VR input
                controller.handle_vr_input(right_goal)
                vr_update_count += 1
            
            # Generate and send action to robot
            action = controller.p_control_action(robot)
            robot.send_action(action)
            
            loop_count += 1
            
            # Status update every 2 seconds
            current_time = time.time()
            if current_time - last_print_time >= 2.0:
                elapsed = current_time - start_time
                vr_status = "📍 VR Connected" if vr_monitor.is_connected() else "⏳ Waiting VR"
                vr_freq = vr_update_count / max(elapsed, 1)
                
                print(f"⏱️  {elapsed:6.1f}s | Loop: {loop_count:6d} ({loop_count/max(elapsed,1):.0f}Hz) | "
                      f"VR Updates: {vr_update_count:5d} ({vr_freq:.1f}Hz) | "
                      f"EE: ({controller.current_x:.4f}, {controller.current_y:.4f}) | {vr_status}")
                
                last_print_time = current_time
            
            # Control loop frequency (50Hz)
            time.sleep(0.02)
            
    except KeyboardInterrupt:
        print("\n⏹️  Stopping control loop...")
    except Exception as e:
        print(f"\n❌ Control error: {e}")
        traceback.print_exc()
    finally:
        print("✅ VR Control Loop ended")

🚀 Starting VR Control Loop...
💡 Press Stop button to exit

📊 Monitoring Status:
----------------------------------------------------------------------
⏱️     2.0s | Loop:     92 (46Hz) | VR Updates:    92 (45.5Hz) | EE: (0.3131, 0.1556) | 📍 VR Connected
⏱️     4.0s | Loop:    182 (45Hz) | VR Updates:   182 (45.2Hz) | EE: (0.3087, 0.1813) | 📍 VR Connected
⏱️     6.0s | Loop:    273 (45Hz) | VR Updates:   273 (45.2Hz) | EE: (0.3069, 0.1864) | 📍 VR Connected
⏱️     8.0s | Loop:    363 (45Hz) | VR Updates:   363 (45.1Hz) | EE: (0.2996, 0.1936) | 📍 VR Connected
⏱️    10.1s | Loop:    454 (45Hz) | VR Updates:   454 (45.1Hz) | EE: (0.3081, 0.1787) | 📍 VR Connected

⏹️  Stopping control loop...
✅ VR Control Loop ended


---

## 🛑 CLEANUP: 안전하게 종료

**로봇과 VR 연결을 안전하게 해제합니다.**

⚠️ 제어를 종료할 때 항상 이 셀을 실행하세요!

## !!셀 실행 시 로봇팔의 토크가 갑자기 풀리니 주의!!

In [ ]:
# Cleanup and Shutdown
print("🔄 Cleaning up...")

robot = globals_dict.get('robot')
vr_monitor = globals_dict.get('vr_monitor')
event_loop_thread = globals_dict.get('event_loop_thread')

try:
    if robot:
        robot.disconnect()
        logger.info("✅ Robot disconnected")
    
    if vr_monitor:
        vr_monitor.is_running = False
        logger.info("✅ VR Monitor stopped")
    
    if event_loop_thread:
        event_loop_thread.stop()
        logger.info("✅ Event loop thread stopped")
    
    # Give threads time to stop
    time.sleep(0.5)
    
    print("\n" + "="*70)
    print("✅ ALL SYSTEMS SAFELY SHUT DOWN")
    print("="*70)
    
except Exception as e:
    logger.error(f"❌ Cleanup error: {e}")
    traceback.print_exc()

# Reset globals
globals_dict['robot'] = None
globals_dict['vr_monitor'] = None
globals_dict['controller'] = None
globals_dict['event_loop_thread'] = None
globals_dict['event_loop'] = None

---

## 📚 QUICK REFERENCE

### 실행 순서
```
1️⃣  라이브러리 임포트 (셀 1-3)
    ↓
2️⃣  SO100 연결 (SECTION 1)
    ↓
3️⃣  VR 서버 시작 (SECTION 2)
    ├─ ✅ HTTPS 웹 UI 서버 (8443) ← 웹 페이지 제공
    ├─ ✅ WebSocket 서버 (8442) ← VR 데이터 수신
    ├─ URL 복사 (https://[IP]:8443)
    ├─ 헤드셋 브라우저에 URL 입력
    ├─ SSL 경고 → Advanced → Continue
    └─ 연결 완료
    ↓
4️⃣  제어 시작 (SECTION 3)
    ├─ VR 모니터링 + 로봇 제어 동시 실행
    ├─ 실시간 상태 출력 (2초마다)
    └─ VR 입력이 계속 받아짐
    ↓
5️⃣  정리 (CLEANUP)
```

### 🎮 컨트롤러 맵핑
| VR 입력 | 로봇 제어 |
|--------|----------|
| 컨트롤러 위치 | 팔의 end-effector 위치 |
| 손목 각도 | 손목 회전 (roll/flex) |
| 트리거 (> 50%) | 그리퍼 개폐 |

### 🔧 서버 아키텍처
```
클라이언트 (MetaQuest)
    ↓
┌─────────────────────────────────┐
│ HTTPS Web Server (8443)         │
│ → 웹 페이지, CSS, JavaScript    │
│ → 자체서명 인증서 사용          │
└─────────────────────────────────┘
    ↓ (사용자 조종)
┌─────────────────────────────────┐
│ WebSocket Server (8442)         │
│ → VR 컨트롤러 데이터 수신       │
└─────────────────────────────────┘
    ↓
┌─────────────────────────────────┐
│ 노트북 (이 파일)                │
│ → 로봇 제어 루프 (50Hz)         │
│ → VR 데이터 처리                │
└─────────────────────────────────┘
```

### 🚨 문제 해결

**"웹에서 접속을 거부했습니다"**
- ✅ URL이 `https://`로 시작하는지 확인 (http://가 아님)
- ✅ 포트 번호가 **8443**인지 확인
- ✅ SECTION 2에서 두 개의 서버 시작 메시지 확인:
  - `HTTPS 웹 UI 서버` ✅
  - `WebSocket 서버` ✅
- ✅ 컴퓨터와 MetaQuest가 같은 네트워크에 있는지 확인
- ✅ SSL 경고가 나오면 **"Advanced" → "Continue"** 클릭 (필수!)

**"SSL 인증서 오류"**
- 정상입니다! 자체 서명된 인증서를 사용합니다
- "신뢰할 수 없음" 경고 후에 **"계속"** 버튼이 있습니다
- 클릭해서 진행하세요 (보안상 위험하지 않습니다)

**"VR 위치가 한 번만 떠요"**
- ✅ SECTION 2에서 asyncio 이벤트 루프가 제대로 시작되었는지 확인
- ✅ SECTION 3 실행 중 "VR Updates" 값이 증가하는지 확인
- ✅ MetaQuest가 여전히 연결되어 있는지 확인

**"SO100 미연결"**
- SECTION 1 재실행 또는 USB 포트 확인 (`/dev/ttyACM0`)

**"MetaQuest 미접속"**
- SECTION 2에서 URL 재확인 후 헤드셋에서 접속
- SSL 경고 시 **Advanced → Continue** 클릭